In [0]:
from pyspark.sql.functions import col, current_timestamp, current_date
from delta.tables import DeltaTable

CATALOG = "dbr_dev_ua5816bd"
LOGIN = "elina_sharabura"

RAW_SCHEMA = LOGIN
BRONZE_SCHEMA = f"{LOGIN}_bronze"

TABLE_NAME = "orders"
TARGET_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.{TABLE_NAME}"

FILE_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/raw_data/orders.csv"
MERGE_KEY = "order_id"

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS dbr_dev_ua5816bd.elina_sharabura.raw_data;

In [0]:
df_raw = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(FILE_PATH)
)

In [0]:
df_bronze = (
    df_raw
    .dropDuplicates(["order_id"])
    .withColumn("source_filename", col("_metadata.file_name"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("load_date", current_date())
)

In [0]:
from delta.tables import DeltaTable

if not spark.catalog.tableExists(TARGET_TABLE):

    (
        df_bronze.write
        .format("delta")
        .saveAsTable(TARGET_TABLE)
    )

    print(f"Created {TARGET_TABLE}")

else:

    delta_table = DeltaTable.forName(spark, TARGET_TABLE)

    (
        delta_table.alias("target")
        .merge(
            df_bronze.alias("source"),
            f"target.{MERGE_KEY} = source.{MERGE_KEY}"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"New records loaded into {TARGET_TABLE}")

In [0]:
print(spark.table(TARGET_TABLE).count())